<a href="https://colab.research.google.com/github/nikvijay07/nlp_final/blob/main/NLP_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Final Project
Authors: Nikhil Vijayvergiya, Aadi Katta, Saahir Shaik, Ethan Fong


## Background
For our final project, we are researching how different models perform on sentences with contrasting conjugations.  

## 1. Setup - Imports

We have imported many of the core libraries as well as HuggingFace and PyTorch which handles the underlying framework of the models and abstract away certain technicalities of the training loop to allow for focus on fine tuning.

In [ ]:
!pip install transformers


In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report
import random

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Check what version of Python is running
print(sys.version)
print(device)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
cuda


## 2. Handling the Data

We have chosen the [SST2 dataset](https://huggingface.co/datasets/stanfordnlp/sst2) for our benchmark. This dataset contains thousands of examples extracted from movie reviews and contains a wide variety of sentences with different structures to allow us for a more comprehensive evaluation.

Since we are focusing on sentences that have complex structure with shifts in their sentiment, we are creating an additional challenge set that only contains sentences with certain contrasting conjugations. We will use this challenge set to evaluate each model on to give us a baseline understanding of how they perform on these sentences.

Below you can see examples of these sentences



In [ ]:
from collections import Counter
from datasets import Dataset

dataset = load_dataset("glue", "sst2")

triggers = ['not', 'but', 'however', 'although', 'despite', 'never', 'hardly', 'though', 'whereas', 'while', 'in contrast', 'on the other hand', 'yet' ]

trigger_counts = Counter()
challenge_examples = []

for ex in dataset['validation']:
    sentence = ex['sentence'].lower()
    matched = [t for t in triggers if t in sentence]

    if matched:
        challenge_examples.append(ex)
        for t in set(matched):
            trigger_counts[t] += 1

challenge_set = Dataset.from_list(challenge_examples)

print(f"Total training examples: {len(dataset['train'])}")
print(f"Total standard validation examples: {len(dataset['validation'])}")
print(f"Challenge Set examples (contains triggers): {len(challenge_set)}")

print("\nTrigger counts:")
print(trigger_counts)

print("\nSample Challenge Set Sentences:")
for i in range(3):
    print(f"Label: {challenge_set[i]['label']} | Text: {challenge_set[i]['sentence']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Total training examples: 67349
Total standard validation examples: 872
Challenge Set examples (contains triggers): 246

Trigger counts:
Counter({'but': 112, 'not': 102, 'though': 26, 'while': 17, 'never': 14, 'despite': 11, 'yet': 8, 'although': 6, 'hardly': 2, 'however': 2})

Sample Challenge Set Sentences:
Label: 1 | Text: allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . 
Label: 1 | Text: although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women . 
Label: 1 | Text: we root for ( clara and paul ) , even like them , though perhaps it 's an emotion closer to pity . 


## 3. Majority Class Baseline

The first evaluation is just a majority class predictor. As you can see, it has scored a 50.92% which shows there is a even split among the data validation set.

In [18]:
# Get the most frequent class in the training set
train_labels = dataset['train']['label']
majority_class = max(set(train_labels), key=train_labels.count)

# Evaluate on validation set
val_labels = dataset['validation']['label']
majority_predictions = [majority_class] * len(val_labels)

print("--- Majority Class Baseline (Validation Set) ---")
print(f"Accuracy: {accuracy_score(val_labels, majority_predictions):.4f}")
print(classification_report(val_labels, majority_predictions, target_names=["Negative", "Positive"], zero_division=0))

# Evaluate on challenge set
challenge_labels = challenge_set['label']
challenge_predictions = [majority_class] * len(challenge_labels)

print("\n--- Majority Class Baseline (Challenge Set) ---")
print(f"Accuracy: {accuracy_score(challenge_labels, challenge_predictions):.4f}")
print(classification_report(
    challenge_labels,
    challenge_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

--- Majority Class Baseline (Validation Set) ---
Accuracy: 0.5092
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       428
    Positive       0.51      1.00      0.67       444

    accuracy                           0.51       872
   macro avg       0.25      0.50      0.34       872
weighted avg       0.26      0.51      0.34       872



## 4. RoBERTa-Base Model

[RoBERTa](https://huggingface.co/FacebookAI/roberta-base) is currently a SOTA model for sentiment analysis. This is a transfomer based model which utilizes attention to capture relationships between words to get a stronger "understanding" of a sentence. The mechanism of attention is the main reason why it has performed very well at sentiment analysis and can even capture shifts in sentence composition.

Below, we are tokenizing the dataset for the model to process and training it.

In [ ]:
model_name = "roberta-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

# Tokenize all datasets
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_challenge = challenge_set.map(tokenize_function, batched=True)

# Load Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {"accuracy": acc, "f1": f1}

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/246 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
training_args = TrainingArguments(
    output_dir="./roberta_sst2_results",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.1,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,   # <-- CHANGED THIS LINE (formerly tokenizer=tokenizer)
    compute_metrics=compute_metrics,
)

# Start training
print("Starting RoBERTa-base training...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting RoBERTa-base training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.387879,0.180577,0.940367,0.940774
2,0.291591,0.202653,0.948394,0.949944
3,0.239909,0.205738,0.943807,0.945006
4,0.212284,0.253006,0.943807,0.945006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=8420, training_loss=0.32377771662986193, metrics={'train_runtime': 1985.0486, 'train_samples_per_second': 135.713, 'train_steps_per_second': 4.242, 'total_flos': 1.772026646744064e+16, 'train_loss': 0.32377771662986193, 'epoch': 4.0})

In [15]:
print("\n--- Evaluating on Standard Validation Set ---")
standard_results = trainer.evaluate(eval_dataset=tokenized_datasets["validation"])
print(standard_results)

print("\n--- Evaluating on Compositional Challenge Set ---")
challenge_results = trainer.evaluate(eval_dataset=tokenized_challenge)
print(challenge_results)

# To get the full classification report for the challenge set:
predictions = trainer.predict(tokenized_challenge)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print("\nChallenge Set Detailed Report:")
print(classification_report(labels, preds, target_names=["Negative", "Positive"]))


--- Evaluating on Standard Validation Set ---


{'eval_loss': 0.1804818958044052, 'eval_accuracy': 0.9426605504587156, 'eval_f1': 0.9431818181818182, 'eval_runtime': 1.8221, 'eval_samples_per_second': 478.558, 'eval_steps_per_second': 30.184, 'epoch': 4.0}

--- Evaluating on Compositional Challenge Set ---
{'eval_loss': 0.20311704277992249, 'eval_accuracy': 0.9390243902439024, 'eval_f1': 0.9315068493150684, 'eval_runtime': 0.5705, 'eval_samples_per_second': 431.216, 'eval_steps_per_second': 28.047, 'epoch': 4.0}

Challenge Set Detailed Report:
              precision    recall  f1-score   support

    Negative       0.93      0.96      0.95       135
    Positive       0.94      0.92      0.93       111

    accuracy                           0.94       246
   macro avg       0.94      0.94      0.94       246
weighted avg       0.94      0.94      0.94       246



### Targeted Sampling for RoBERTa
To improve RoBERTa's performance on the challenge set, we can perform targeted sampling. This means that we are training it on the original dataset ***plus*** a filtered dataset of the original dataset with examples containing triggers. This essentially adds more weight to those examples and will hopefully allow the model to perform better on these more complex sentences, without sacrificing accuracy on the general validation set.

In [17]:
# Identify trigger-containing examples within the original training dataset
# We'll re-use the 'triggers' list defined earlier.
from datasets import Dataset

train_trigger_examples = []
for ex in dataset['train']:
    sentence = ex['sentence'].lower()
    matched = [t for t in triggers if t in sentence]
    if matched:
        train_trigger_examples.append(ex)

# Convert to a Hugging Face Dataset
train_trigger_dataset = Dataset.from_list(train_trigger_examples)
print(f"Found {len(train_trigger_dataset)} trigger-containing examples in the original training set.")

# Tokenize these examples
tokenized_train_trigger = train_trigger_dataset.map(tokenize_function, batched=True)
print("Tokenized trigger-containing training examples.")

# Combine the original training set with these identified examples for augmented training
# We'll concatenate them, effectively giving more weight to the trigger-containing examples.
augmented_train_dataset = torch.utils.data.ConcatDataset([
    tokenized_datasets['train'],
    tokenized_train_trigger
])

print(f"Original training set size: {len(tokenized_datasets['train'])}")
print(f"Augmented training set size: {len(augmented_train_dataset)}")

# --- Create a NEW RoBERTa model for augmented training ---
print("Loading a fresh RoBERTa-base model for augmented training...")
augmented_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Re-initializing Trainer with augmented training data...")
augmented_trainer = Trainer(
    model=augmented_model, # Use the NEWLY loaded RoBERTa model
    args=training_args, # Use existing training arguments
    train_dataset=augmented_train_dataset,
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting RoBERTa-base training with augmented data...")
augmented_trainer.train()

# Evaluate the retrained model on both validation and challenge sets
print("\n--- Evaluating RoBERTa (Augmented Training) on Standard Validation Set ---")
augmented_standard_results = augmented_trainer.evaluate(eval_dataset=tokenized_datasets["validation"])
print(augmented_standard_results)

print("\n--- Evaluating RoBERTa (Augmented Training) on Compositional Challenge Set ---")
augmented_challenge_results = augmented_trainer.evaluate(eval_dataset=tokenized_challenge)
print(augmented_challenge_results)

# Get detailed classification report for the augmented challenge set evaluation
augmented_predictions = augmented_trainer.predict(tokenized_challenge)
augmented_preds = np.argmax(augmented_predictions.predictions, axis=-1)
augmented_labels = augmented_predictions.label_ids
print("\nChallenge Set Detailed Report (Augmented Training):")
print(classification_report(augmented_labels, augmented_preds, target_names=["Negative", "Positive"]))

Found 7627 trigger-containing examples in the original training set.


Map:   0%|          | 0/7627 [00:00<?, ? examples/s]

Tokenized trigger-containing training examples.
Original training set size: 67349
Augmented training set size: 74976
Loading a fresh RoBERTa-base model for augmented training...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.387212,0.181140,0.936927,0.939227
2,0.274959,0.243557,0.941514,0.943270
3,0.223861,0.230035,0.944954,0.946429
4,0.181459,0.276817,0.944954,0.946429


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


--- Evaluating RoBERTa (Augmented Training) on Standard Validation Set ---


{'eval_loss': 0.18122495710849762, 'eval_accuracy': 0.9369266055045872, 'eval_f1': 0.9392265193370166, 'eval_runtime': 1.8042, 'eval_samples_per_second': 483.317, 'eval_steps_per_second': 30.484, 'epoch': 4.0}

--- Evaluating RoBERTa (Augmented Training) on Compositional Challenge Set ---
{'eval_loss': 0.1967250555753708, 'eval_accuracy': 0.926829268292683, 'eval_f1': 0.9196428571428571, 'eval_runtime': 0.5264, 'eval_samples_per_second': 467.344, 'eval_steps_per_second': 30.396, 'epoch': 4.0}

Challenge Set Detailed Report (Augmented Training):
              precision    recall  f1-score   support

    Negative       0.94      0.93      0.93       135
    Positive       0.91      0.93      0.92       111

    accuracy                           0.93       246
   macro avg       0.93      0.93      0.93       246
weighted avg       0.93      0.93      0.93       246



### GloVe Embeddings

Before we implement our BiLSTM model, we would like to use GloVe embeddings. GloVe embeddings have already captured semantic meaning within their vectors and our pretrained, which is very helpful for our case.

In [8]:
!wget https://raw.githubusercontent.com/aritter/aritter.github.io/master/files/glove.840B.300d.conll_filtered.txt



--2026-05-02 02:22:46--  https://raw.githubusercontent.com/aritter/aritter.github.io/master/files/glove.840B.300d.conll_filtered.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 69798443 (67M) [text/plain]
Saving to: ‘glove.840B.300d.conll_filtered.txt’

glove.840B.300d.con 100%[===================>]  66.56M   366MB/s    in 0.2s    

2026-05-02 02:22:48 (366 MB/s) - ‘glove.840B.300d.conll_filtered.txt’ saved [69798443/69798443]



In [9]:
glove_path = 'glove.840B.300d.conll_filtered.txt'

def GloVe_to_dict(path):
    embeddings = {}
    for i in open(path).readlines():
      line = i.strip().split(" ")
      if len(line) == 0:
        continue
      word, vectors = line[0], line[1:]
      embeddings[word] = [float(x) for x in vectors]
    return embeddings

glove_embeddings = GloVe_to_dict(glove_path)
print(f"Loaded {len(glove_embeddings)} GloVe embeddings.")
print(glove_embeddings["where"])


Loaded 26905 GloVe embeddings.
[0.44746, 0.18366, -0.33118, -0.22959, 0.45431, 0.20785, -0.19588, 0.0057302, 0.060887, 2.9707, -0.23427, -0.037312, -0.1093, 0.10037, -0.11392, -0.12324, 0.14923, 1.0743, -0.25514, -0.26972, 0.21964, 0.13038, -0.2155, -0.090606, 0.038431, -0.32106, -0.084176, -0.05368, -0.11713, 0.21232, -0.06269, -0.02708, 0.24056, -0.13784, -0.081213, -0.023957, 0.093773, 0.17192, -0.085551, -0.32483, -0.08996, -0.097737, 0.43865, -0.29887, -0.087982, -0.064557, -0.36973, -0.33135, -0.070849, -0.14902, 0.20335, 0.30167, -0.08412, 0.29771, 0.22192, -0.18107, 0.059249, -0.46247, 0.0050414, 0.11282, -0.085125, 0.077902, 0.25767, 0.38706, -0.13504, -0.058158, 0.24725, 0.18287, 0.029546, 0.017681, 0.15403, 0.019649, 0.43471, 0.11392, 0.031199, 0.12942, 0.013383, -0.0073397, 0.18861, 0.082347, 0.27742, 0.21251, -0.2612, -0.17046, -0.040688, -0.12544, -0.26735, -0.17528, 0.23339, 0.12942, -0.36945, -0.051172, 0.021642, -0.15552, 0.04742, -0.070481, -0.13322, 0.082576, -0.0027

## 5. Bi-LSTM Model

Our second model is a Bi directional LSTM model. This is a RNN model that reads each sentence from left to right and then from right to left. This allows it to hold context from both sides of the sentence when extracting its meaning.

This model has 4 layers:
 an embedding layer which contains all the

1.   Embedding Layer: contains all the GloVe pre-trained embeddings
2.   Bidirectional LSTM Layer:
3.    Dropout Layer: Prevents overfitting by randomly setting certain neuron's weights to 0
4.   Linear Layer: Reduces the number of output classes to 2 to produce the final classification


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np
import os


# --- 1. Define the Bi-LSTM Model ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers, dropout_rate, pretrained_embeddings=None):
        super(BiLSTMClassifier, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Bi-LSTM layer with dropout between layers
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout_rate if num_layers > 1 else 0)

        self.dropout = nn.Dropout(dropout_rate)

        # Output layer (hidden_dim * 2 because of bidirectional LSTM)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, input_ids):
        # input_ids: (batch_size, sequence_length)

        embedded = self.embedding(input_ids) # (batch_size, sequence_length, embedding_dim)

        # Initialize hidden and cell states
        # h_0 and c_0: (num_layers * num_directions, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers * 2, input_ids.size(0), self.hidden_dim).to(input_ids.device)
        c0 = torch.zeros(self.num_layers * 2, input_ids.size(0), self.hidden_dim).to(input_ids.device)

        # out: (batch_size, sequence_length, hidden_dim * num_directions)
        # h_n, c_n: (num_layers * num_directions, batch_size, hidden_dim)
        out, (h_n, c_n) = self.lstm(embedded, (h0, c0))

        final_hidden_state = self.dropout(torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1))

        logits = self.fc(final_hidden_state)
        return logits


In [13]:
# 2. Custom Dataset for Bi-LSTM

class BiLSTMDataset(Dataset):
    def __init__(self, tokenized_data):
        self.tokenized_data = tokenized_data

    def __len__(self):
        return len(self.tokenized_data['input_ids'])

    def __getitem__(self, idx):
        # Ensure the items are converted to tensors
        input_ids = torch.tensor(self.tokenized_data['input_ids'][idx], dtype=torch.long)
        labels = torch.tensor(self.tokenized_data['label'][idx], dtype=torch.long)
        return input_ids, labels

# 3. Hyperparameters

BILSTM_VOCAB_SIZE = tokenizer.vocab_size
BILSTM_EMBEDDING_DIM = 300
BILSTM_HIDDEN_DIM = 256
BILSTM_OUTPUT_DIM = 2 # Since it is binary classification
BILSTM_NUM_LAYERS = 2
BILSTM_DROPOUT_RATE = 0.35
BILSTM_BATCH_SIZE = 16
BILSTM_LEARNING_RATE = 1e-5

BILSTM_NUM_EPOCHS = 5
BILSTM_GRAD_NORM_CLIP = 1.0

print("Creating embedding matrix from pre-loaded GloVe embeddings...")

try:
    glove_embedding_matrix = torch.zeros((BILSTM_VOCAB_SIZE, BILSTM_EMBEDDING_DIM))
    words_found = 0

    # Map tokens from current tokenizer to GloVe embeddings
    for i in range(BILSTM_VOCAB_SIZE):
        token = tokenizer.convert_ids_to_tokens(i)

        # Handle special tokens (e.g., 'Ġ' for RoBERTa) or lowercasing
        token_processed = token.replace('Ġ', '').lower() # Basic processing for RoBERTa tokens
        if token_processed in glove_embeddings:
            glove_embedding_matrix[i] = torch.tensor(glove_embeddings[token_processed])
            words_found += 1
        else:

            # Initialize out-of-vocabulary words randomly
            glove_embedding_matrix[i] = torch.rand(BILSTM_EMBEDDING_DIM) * 0.1 # Small random values
    print(f"Initialized {words_found}/{BILSTM_VOCAB_SIZE} words with pre-trained GloVe embeddings.")
except Exception as e:
    print(f"Could not create embedding matrix from custom GloVe dictionary: {e}. Initializing embedding matrix randomly.")
    glove_embedding_matrix = None # Fallback to random initialization


# Instantiate the model and move it to the appropriate device (cuda or cpu)
bilstm_model = BiLSTMClassifier(
    vocab_size=BILSTM_VOCAB_SIZE,
    embedding_dim=BILSTM_EMBEDDING_DIM,
    hidden_dim=BILSTM_HIDDEN_DIM,
    output_dim=BILSTM_OUTPUT_DIM,
    num_layers=BILSTM_NUM_LAYERS,
    dropout_rate=BILSTM_DROPOUT_RATE,
    pretrained_embeddings=glove_embedding_matrix if glove_embedding_matrix is not None else None # Pass GloVe embeddings
).to(device)

optimizer = optim.Adam(bilstm_model.parameters(), lr=BILSTM_LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

Creating embedding matrix from pre-loaded GloVe embeddings...
Initialized 17855/50265 words with pre-trained GloVe embeddings.


In [14]:

# --- 4. Prepare DataLoaders ---
# 'tokenized_datasets' and 'tokenized_challenge' are expected to be available from previous cells
train_bilstm_dataset = BiLSTMDataset(tokenized_datasets['train'])
val_bilstm_dataset = BiLSTMDataset(tokenized_datasets['validation'])
challenge_bilstm_dataset = BiLSTMDataset(tokenized_challenge)

train_loader = DataLoader(train_bilstm_dataset, batch_size=BILSTM_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_bilstm_dataset, batch_size=BILSTM_BATCH_SIZE, shuffle=False)
challenge_loader = DataLoader(challenge_bilstm_dataset, batch_size=BILSTM_BATCH_SIZE, shuffle=False)


def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)

        optimizer.zero_grad() # Clear gradients

        logits = model(input_ids) # Forward pass
        loss = criterion(logits, labels) # Compute loss

        loss.backward() # Backward pass (compute gradients)
        nn.utils.clip_grad_norm_(model.parameters(), BILSTM_GRAD_NORM_CLIP) # New: Gradient clipping
        optimizer.step() # Update model parameters

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval() # Set model to evaluation mode
    all_preds = []
    all_labels = []
    with torch.no_grad(): # Disable gradient calculation for evaluation
        for input_ids, labels in dataloader:
            input_ids, labels = input_ids.to(device), labels.to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=1).cpu().numpy() # Get predicted class indices

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='binary') # SST-2 is binary classification
    return acc, f1, all_labels, all_preds

print(f"\n--- Starting Bi-LSTM Training for {BILSTM_NUM_EPOCHS} epochs ---")
for epoch in range(BILSTM_NUM_EPOCHS):
    train_loss = train_epoch(bilstm_model, train_loader, optimizer, criterion)
    val_acc, val_f1, _, _ = evaluate_model(bilstm_model, val_loader)
    print(f"Epoch {epoch+1}/{BILSTM_NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

# --- 6. Evaluate on Challenge Set and Main Dataset ---
print("\n--- Evaluating Bi-LSTM on Standard Validation Set ---")
val_acc, val_f1, val_true_labels, val_pred_labels = evaluate_model(bilstm_model, val_loader)
print(f"Accuracy: {val_acc:.4f}")
print(f"F1 Score: {val_f1:.4f}")
print("\nDetailed Report (Validation Set):")
print(classification_report(val_true_labels, val_pred_labels, target_names=["Negative", "Positive"]))

print("\n--- Evaluating Bi-LSTM on Compositional Challenge Set ---")
challenge_acc, challenge_f1, challenge_true_labels, challenge_pred_labels = evaluate_model(bilstm_model, challenge_loader)
print(f"Accuracy: {challenge_acc:.4f}")
print(f"F1 Score: {challenge_f1:.4f}")
print("\nDetailed Report (Challenge Set):")
print(classification_report(challenge_true_labels, challenge_pred_labels, target_names=["Negative", "Positive"]))



--- Starting Bi-LSTM Training for 5 epochs ---
Epoch 1/5 | Train Loss: 0.5955 | Val Acc: 0.7477 | Val F1: 0.7645
Epoch 2/5 | Train Loss: 0.5013 | Val Acc: 0.7603 | Val F1: 0.7765
Epoch 3/5 | Train Loss: 0.4544 | Val Acc: 0.7580 | Val F1: 0.7917
Epoch 4/5 | Train Loss: 0.4161 | Val Acc: 0.8062 | Val F1: 0.8120
Epoch 5/5 | Train Loss: 0.3812 | Val Acc: 0.8096 | Val F1: 0.8092

--- Evaluating Bi-LSTM on Standard Validation Set ---
Accuracy: 0.8096
F1 Score: 0.8092

Detailed Report (Validation Set):
              precision    recall  f1-score   support

    Negative       0.79      0.83      0.81       428
    Positive       0.83      0.79      0.81       444

    accuracy                           0.81       872
   macro avg       0.81      0.81      0.81       872
weighted avg       0.81      0.81      0.81       872


--- Evaluating Bi-LSTM on Compositional Challenge Set ---
Accuracy: 0.7602
F1 Score: 0.7306

Detailed Report (Challenge Set):
              precision    recall  f1-score 

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# --- 1. Define the Neural Bag of Words (NBOW) Model ---
class NBOWClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, pretrained_embeddings=None, dropout_rate=0.0):
        super(NBOWClassifier, self).__init__()

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.dropout = nn.Dropout(dropout_rate)
        # Output layer for classification
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, input_ids):
        # input_ids: (batch_size, sequence_length)

        embedded = self.embedding(input_ids) # (batch_size, sequence_length, embedding_dim)

        # Sum/average embeddings along the sequence dimension (Bag-of-Words approach)
        # We'll use mean pooling for simplicity and robustness to varying sequence lengths
        sentence_embedding = embedded.mean(dim=1) # (batch_size, embedding_dim)

        sentence_embedding = self.dropout(sentence_embedding)
        logits = self.fc(sentence_embedding)
        return logits

# 2. Hyperparameters and Initialization

NBOW_VOCAB_SIZE = tokenizer.vocab_size
NBOW_EMBEDDING_DIM = 300
NBOW_OUTPUT_DIM = 2 # Since it is binary classification
NBOW_LEARNING_RATE = 1e-3
NBOW_NUM_EPOCHS = 3

nbow_model = NBOWClassifier(
    vocab_size=NBOW_VOCAB_SIZE,
    embedding_dim=NBOW_EMBEDDING_DIM,
    output_dim=NBOW_OUTPUT_DIM,
    pretrained_embeddings=glove_embedding_matrix if glove_embedding_matrix is not None else None,
).to(device)

optimizer_nbow = optim.Adam(nbow_model.parameters(), lr=NBOW_LEARNING_RATE)
criterion_nbow = nn.CrossEntropyLoss()

# --- 3. Prepare DataLoaders (re-using existing ones) ---
# train_loader, val_loader, challenge_loader are already defined from previous cells

# --- 4. Training and Evaluation Loops (re-using existing functions) ---
# NUM_EPOCHS and GRAD_NORM_CLIP are already defined

print(f"\n--- Starting NBOW Training for {NBOW_NUM_EPOCHS} epochs ---")
for epoch in range(NBOW_NUM_EPOCHS):
    train_loss_nbow = train_epoch(nbow_model, train_loader, optimizer_nbow, criterion_nbow)
    val_acc_nbow, val_f1_nbow, _, _ = evaluate_model(nbow_model, val_loader)
    print(f"Epoch {epoch+1}/{NBOW_NUM_EPOCHS} | Train Loss: {train_loss_nbow:.4f} | Val Acc: {val_acc_nbow:.4f} | Val F1: {val_f1_nbow:.4f}")

# --- 5. Evaluate on Challenge Set and Main Dataset ---
print("\n--- Evaluating NBOW on Standard Validation Set ---")
val_acc_nbow, val_f1_nbow, val_true_labels_nbow, val_pred_labels_nbow = evaluate_model(nbow_model, val_loader)
print(f"Accuracy: {val_acc_nbow:.4f}")
print(f"F1 Score: {val_f1_nbow:.4f}")
print("\nDetailed Report (Validation Set):")
print(classification_report(val_true_labels_nbow, val_pred_labels_nbow, target_names=["Negative", "Positive"]))

print("\n--- Evaluating NBOW on Compositional Challenge Set ---")
challenge_acc_nbow, challenge_f1_nbow, challenge_true_labels_nbow, challenge_pred_labels_nbow = evaluate_model(nbow_model, challenge_loader)
print(f"Accuracy: {challenge_acc_nbow:.4f}")
print(f"F1 Score: {challenge_f1_nbow:.4f}")
print("\nDetailed Report (Challenge Set):")
print(classification_report(challenge_true_labels_nbow, challenge_pred_labels_nbow, target_names=["Negative", "Positive"]))


--- Starting NBOW Training for 3 epochs ---
Epoch 1/3 | Train Loss: 0.4914 | Val Acc: 0.8016 | Val F1: 0.8138
Epoch 2/3 | Train Loss: 0.3010 | Val Acc: 0.8234 | Val F1: 0.8319
Epoch 3/3 | Train Loss: 0.2542 | Val Acc: 0.8131 | Val F1: 0.8249

--- Evaluating NBOW on Standard Validation Set ---
Accuracy: 0.8131
F1 Score: 0.8249

Detailed Report (Validation Set):
              precision    recall  f1-score   support

    Negative       0.84      0.76      0.80       428
    Positive       0.79      0.86      0.82       444

    accuracy                           0.81       872
   macro avg       0.82      0.81      0.81       872
weighted avg       0.82      0.81      0.81       872


--- Evaluating NBOW on Compositional Challenge Set ---
Accuracy: 0.7724
F1 Score: 0.7686

Detailed Report (Challenge Set):
              precision    recall  f1-score   support

    Negative       0.84      0.72      0.78       135
    Positive       0.71      0.84      0.77       111

    accuracy         